# SC-SSTW minimal GPU observation probe

Runs only a saved-video relation observation probe. This is not detection evidence, fixed-FPR evidence, or a paper claim.

In [ ]:
# Cell 1: mount Google Drive and define output root
from google.colab import drive
from pathlib import Path
import datetime, os
drive.mount('/content/drive')
RUN_ID = datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/SSTW/diagnostic_tests/sc_sstw_gpu_observation_probe') / RUN_ID
LOCAL_OUTPUT_DIR = Path('/content/sc_sstw_gpu_observation_probe_output')
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('run_id =', RUN_ID)
print('drive_output_root =', DRIVE_OUTPUT_ROOT)


In [ ]:
# Cell 2: clone or update repo, show commit/ref
import os, pathlib, subprocess
repo_dir = pathlib.Path('/content/SC-SSTW-Feasibility')
repo_url = os.environ.get('SC_SSTW_REPO_URL', 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git')
repo_ref = os.environ.get('SC_SSTW_REF', 'main')
if repo_dir.exists():
    subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=repo_dir, check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', 'checkout', repo_ref], cwd=repo_dir, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo_dir, text=True).strip()
print('repo_url =', repo_url)
print('repo_ref =', repo_ref)
print('commit =', commit)


In [ ]:
# Cell 3: install/check minimal dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.2', 'transformers', 'accelerate', 'safetensors', 'imageio', 'imageio-ffmpeg'], check=True)
import torch
print('torch =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('This notebook requires a Colab GPU runtime')
print('gpu =', torch.cuda.get_device_name(0))


In [ ]:
# Cell 4: run GPU observation probe
import pathlib, subprocess, sys, shutil
if LOCAL_OUTPUT_DIR.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIR)
cmd = [sys.executable, 'experiments/run_gpu_observation_probe.py', '--output-dir', str(LOCAL_OUTPUT_DIR)]
print('running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
# Cell 5: print JSON summary and save ZIP to Google Drive
import json, pathlib, shutil
result_path = LOCAL_OUTPUT_DIR / 'gpu_observation_probe_result.json'
failure_path = LOCAL_OUTPUT_DIR / 'gpu_observation_probe_failure.json'
path = result_path if result_path.exists() else failure_path
payload = json.loads(path.read_text(encoding='utf-8'))
print('result_file =', path)
print(json.dumps(payload.get('summary', payload), ensure_ascii=False, indent=2, sort_keys=True))
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
drive_json = DRIVE_OUTPUT_ROOT / path.name
shutil.copy2(path, drive_json)
archive_base = '/content/sc_sstw_gpu_observation_probe_' + RUN_ID
archive_path = pathlib.Path(shutil.make_archive(archive_base, 'zip', LOCAL_OUTPUT_DIR))
drive_zip = DRIVE_OUTPUT_ROOT / archive_path.name
shutil.copy2(archive_path, drive_zip)
print('drive_json =', drive_json)
print('drive_zip =', drive_zip)
